In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
import joblib


#Load Dataset
df = pd.read_csv("StudentsPerformance.csv")
df.head()


#Create Target Column and You are predicting average score
df["final_performance"] = (
    df["math score"] +
    df["reading score"] +
    df["writing score"]
) / 3

df.head()


#Select Features & Target
#X = input features
#y = output you want to predict
X = df[
    [
        "gender",
        "race/ethnicity",
        "parental level of education",
        "lunch",
        "test preparation course",
        "math score",
        "reading score",
        "writing score"
    ]
]

y = df["final_performance"]


#Define Categorical Columns/These are text-based columns (not numbers)
categorical_cols = [
    "gender",
    "race/ethnicity",
    "parental level of education",
    "lunch",
    "test preparation course"
]


#Build Preprocessor/OneHotEncoder converts them into numbers,These are text values, so ML cannot understand them directly.
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
    ],
    remainder="passthrough"
)


#Transform Data/Converts full dataset into ML-ready format
X_transformed = preprocessor.fit_transform(X)


#Train-Test Split(80% training, 20% testing)
X_train, X_test, y_train, y_test = train_test_split(
    X_transformed,
    y,
    test_size=0.2,
    random_state=42
)

#Train Model /Learns pattern between inputs → performance score
model = RandomForestRegressor(
    n_estimators=500,
    random_state=42
)

model.fit(X_train, y_train)


#Predictions
y_pred = model.predict(X_test)

#Accuracy Score
r2 = r2_score(y_test, y_pred)
print("R² Score:", r2)


#Save Model + Preprocessor/ These files are used in Flask app
joblib.dump(model, "model.pkl")
joblib.dump(preprocessor, "preprocessor.pkl")

print("Model saved successfully!")



R² Score: 0.9941281689604271
Model saved successfully!


In [8]:
sample = pd.DataFrame([{
    "gender": "female",
    "race/ethnicity": "group B",
    "parental level of education": "bachelor's degree",
    "lunch": "standard",
    "test preparation course": "none",
    "math score": 70,
    "reading score": 80,
    "writing score": 75
}])

sample_transformed = preprocessor.transform(sample)
prediction = model.predict(sample_transformed)

print("Predicted Final Performance:", prediction[0])

Predicted Final Performance: 74.31400000000004
